In [1]:
import pandas as pd
import math

In [2]:
file_path = "1310425495534_tennis_numerical_50_samples.csv"
df = pd.read_csv(file_path)

In [5]:
print("Dataset shape", df.shape)
display(df.head())

Dataset shape (50, 4)


,Temperature_C,Humidity_pct,Wind_kmh,Play
0,22,45,12,No
1,24,50,15,No
2,25,55,18,No
3,27,60,20,Yes
4,29,65,22,Yes


In [6]:
print("Columns: ")
print(df.columns.tolist())

Columns: 
['Temperature_C', 'Humidity_pct', 'Wind_kmh', 'Play']


In [9]:
print("Columns:")
print(df.columns.tolist())


Columns:
['Temperature_C', 'Humidity_pct', 'Wind_kmh', 'Play']


In [10]:
print("\nClass distribution:")
print(df["Play"].value_counts())


Class distribution:
Play
Yes    30
No     20
Name: count, dtype: int64


In [11]:
alpha = 1
N = len(df)
K = df["Play"].nunique()

priors = {}

for c in ["Yes", "No"]:
    Nc = len(df[df["Play"] == c])
    priors[c] = (Nc + alpha) / (N + alpha * K)

print("Class Priors:")
for c, p in priors.items():
    print(f"P({c}) = {p:.6f}")

Class Priors:
P(Yes) = 0.596154
P(No) = 0.403846


In [12]:
features = [
    "Temperature_C",
    "Humidity_pct",
    "Wind_kmh"
]

stats = {}

for c in ["Yes", "No"]:
    
    class_data = df[df["Play"] == c]
    
    stats[c] = {}
    
    for feature in features:
        
        mean = class_data[feature].mean()
        variance = class_data[feature].var(ddof=0)
        
        stats[c][feature] = {
            "mean": mean,
            "variance": variance
        }

# Display results
for c in ["Yes", "No"]:
    print(f"\nClass: {c}")
    
    for feature in features:
        mean = stats[c][feature]["mean"]
        variance = stats[c][feature]["variance"]
        
        print(
            f"{feature}: "
            f"Mean = {mean:.4f}, "
            f"Variance = {variance:.4f}"
        )


Class: Yes
Temperature_C: Mean = 30.8667, Variance = 8.7822
Humidity_pct: Mean = 67.6000, Variance = 55.5067
Wind_kmh: Mean = 24.6333, Variance = 19.1656

Class: No
Temperature_C: Mean = 21.8000, Variance = 5.8600
Humidity_pct: Mean = 45.2000, Variance = 39.6600
Wind_kmh: Mean = 12.5000, Variance = 9.2500


In [13]:
def gaussian_probability(x, mean, variance):
    
    coefficient = 1 / math.sqrt(
        2 * math.pi * variance
    )
    
    exponent = math.exp(
        -((x - mean) ** 2) / (2 * variance)
    )
    
    return coefficient * exponent

In [14]:
test_temperature = 30
test_humidity = 70
test_wind = 25

In [15]:
test = {
    "Temperature_C": 30,
    "Humidity_pct": 70,
    "Wind_kmh": 25
}

In [16]:
p_temp_yes = gaussian_probability(
    30,
    stats["Yes"]["Temperature_C"]["mean"],
    stats["Yes"]["Temperature_C"]["variance"]
)

p_humidity_yes = gaussian_probability(
    70,
    stats["Yes"]["Humidity_pct"]["mean"],
    stats["Yes"]["Humidity_pct"]["variance"]
)

p_wind_yes = gaussian_probability(
    25,
    stats["Yes"]["Wind_kmh"]["mean"],
    stats["Yes"]["Wind_kmh"]["variance"]
)

print("P(Temperature | Yes) =", p_temp_yes)
print("P(Humidity | Yes)    =", p_humidity_yes)
print("P(Wind | Yes)        =", p_wind_yes)

P(Temperature | Yes) = 0.12898406245014585
P(Humidity | Yes)    = 0.050839800364250315
P(Wind | Yes)        = 0.09080841457552054


In [17]:
score_yes = (
    priors["Yes"]
    * p_temp_yes
    * p_humidity_yes
    * p_wind_yes
)

print("Score(Yes) =", score_yes)

Score(Yes) = 0.0003549967126069995


In [18]:
p_temp_no = gaussian_probability(
    30,
    stats["No"]["Temperature_C"]["mean"],
    stats["No"]["Temperature_C"]["variance"]
)

p_humidity_no = gaussian_probability(
    70,
    stats["No"]["Humidity_pct"]["mean"],
    stats["No"]["Humidity_pct"]["variance"]
)

p_wind_no = gaussian_probability(
    25,
    stats["No"]["Wind_kmh"]["mean"],
    stats["No"]["Wind_kmh"]["variance"]
)

print("P(Temperature | No) =", p_temp_no)
print("P(Humidity | No)    =", p_humidity_no)
print("P(Wind | No)        =", p_wind_no)

P(Temperature | No) = 0.0005312835428763844
P(Humidity | No)    = 2.718029448661844e-05
P(Wind | No)        = 2.817159788048395e-05


In [19]:
score_no = (
    priors["No"]
    * p_temp_no
    * p_humidity_no
    * p_wind_no
)

print("Score(No) =", score_no)

Score(No) = 1.6428879829129486e-13


In [20]:
if score_yes > score_no:
    prediction = "Yes"
else:
    prediction = "No"

print("Prediction:", prediction)

Prediction: Yes
